# SCLDO - Supply Chain Late-Delivery & SLA Risk Intelligence

**ElmasCore Data Science Monograph - "From Data to Systems"**

Ce notebook de référence reproduit et documente l'étude complète sur les **96 470 commandes livrées** d'Olist (9 tables relationnelles brésiliennes) :
1. **Ingestion relationnelle & zéro fuite pré-expédition** (distances géodésiques Haversine + historique vendeur strictement passé).
2. **Validation temporelle stricte** (5-fold expanding `TimeSeriesSplit` + OOT holdout `> 2018-04-30`, calibration isotonique OOF imbriquée et ablation de fuite post-transporteur).
3. **Économétrie causale retard → avis 1 étoile** (G-computation logistique + IPW stabilisé + IC bootstrap 95 % sur 95 824 avis).
4. **Politique d'intervention SLA proactive** sous matrice de coût asymétrique `8:1` et attribution TreeSHAP.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results.json"
IMAGES_DIR = PROJECT_ROOT / "images"

if not RESULTS_PATH.exists():
    from main import run_scldo_study
    from src.core.config import PipelineConfig
    run_scldo_study(PipelineConfig.create_default(PROJECT_ROOT))

results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
pd.Series(results["audit"]).to_frame("Value")

## 1. Benchmark Temporel (5-Fold Expanding CV vs Strict OOT Holdout) & Ablation de Fuite Transporteur

In [ ]:
bench_rows = [
    {
        "Regime": r["regime"],
        "Model": r["model_name"],
        "Calibration": r["calibration_method"],
        "CV ROC-AUC": round(r["cv_metrics"]["roc_auc"], 4),
        "CV ECE-10": round(r["cv_metrics"]["ece_10"], 4),
        "OOT ROC-AUC": round(r["oot_metrics"]["roc_auc"], 4),
        "OOT PR-AUC": round(r["oot_metrics"]["pr_auc"], 4),
        "OOT Brier": round(r["oot_metrics"]["brier_score"], 4),
        "OOT ECE-10": round(r["oot_metrics"]["ece_10"], 4),
    }
    for r in results["temporal_benchmark"]["records"]
]
display(pd.DataFrame(bench_rows))
display(Image(filename=str(IMAGES_DIR / "scldo_temporal_model_benchmark.png")))

## 2. Attribution TreeSHAP des Déterminants Pré-Expédition du Retard

In [ ]:
shap_df = pd.DataFrame(results["shap_drivers"]["drivers"])
display(shap_df)
display(Image(filename=str(IMAGES_DIR / "scldo_shap_delay_drivers.png")))

## 3. Économétrie Causale : Impact du Retard sur la Probabilité d'Avis 1 Étoile

In [ ]:
causal_df = pd.DataFrame(results["causal_review_impact"]["buckets"])
display(causal_df[[
    "bucket_label", "sample_count", "mean_review_score",
    "unadjusted_one_star_rate", "adjusted_one_star_rate",
    "risk_diff_vs_ontime", "risk_diff_ci_low", "risk_diff_ci_high", "ipw_one_star_rate"
]])
display(Image(filename=str(IMAGES_DIR / "scldo_causal_delay_reviews.png")))

## 4. Optimisation Économique de l'Intervention SLA Proactive (Ratio Asymétrique 8:1)

In [ ]:
sla = results["sla_policy"]
policy_summary = pd.DataFrame([
    {"Policy": "Default (tau=0.50)", **sla["default_policy"]},
    {"Policy": "Theoretical Bayes (tau=0.125)", **sla["bayes_policy"]},
    {"Policy": "Empirical Optimal (tau=0.130)", **sla["empirical_optimal_policy"]},
])
display(policy_summary)
display(Image(filename=str(IMAGES_DIR / "scldo_sla_policy_economics.png")))